# NCM research run: `calib_t4`

Throughput calibration: 300 steps each of tokenizer / NCM generator / AR at the 'm' scale. Measures real TFLOPS on this GPU so all later estimates are grounded.

- Recommended GPU: **T4**. Estimated cost: **0.6 compute units**.
- Runtime > Change runtime type > GPU, then Runtime > Run all.
- If Colab disconnects, just Run all again: every command resumes from its last Drive checkpoint.
- When done, copy `MyDrive/ncm-research/results/calib_t4/` into the repo's `results/colab/`.

In [ ]:
import os, subprocess, sys, json, shutil, time
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ncm-research'
R = f'{DRIVE}/runs'; D = f'{DRIVE}/data/tinystories'
os.makedirs(R, exist_ok=True); os.makedirs(D, exist_ok=True)
REPO, BRANCH = 'https://github.com/amritisepic/NCM-research.git', 'claude/ncm-research'
if not os.path.exists('/content/NCM-research'):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO, '/content/NCM-research'], check=True)
else:
    subprocess.run(f'cd /content/NCM-research && git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}', shell=True, check=True)
os.chdir('/content/NCM-research')
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf', 'hydra-core', 'regex', 'mauve-text', 'faiss-cpu'], check=True)

In [ ]:
import urllib.request
B = 'https://huggingface.co/datasets/hazyresearch/ncm-datasets/resolve/main/tinystories/'
for f in ['bpe_vocab_8192.pickle', 'bpe_merges_8192.pickle', 'valid_bpe_8192.pickle',
          'train_bpe_8192_offsets.npy', 'train_bpe_8192.bin']:
    p = f'{D}/{f}'
    if not os.path.exists(p):
        print('downloading', f, flush=True)
        urllib.request.urlretrieve(B + f, p + '.tmp'); os.replace(p + '.tmp', p)
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/tinystories'):
    os.symlink(D, 'data/tinystories')
# local copy of the big stream for fast random access (Drive reads are slow)
if not os.path.exists('/content/ts'):
    os.makedirs('/content/ts');
    for f in os.listdir(D): shutil.copy(f'{D}/{f}', f'/content/ts/{f}')
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
CMDS = [
 "python scripts/train.py tokenizer --config configs/calib/tok_m.yaml --run_dir {R}/calib_tok_m",
 "python scripts/train.py ar --config configs/calib/ar_m.yaml --run_dir {R}/calib_ar_m",
 "python scripts/train.py ncm --config configs/calib/ncm_m.yaml --tokenizer {R}/calib_tok_m/final.pt --run_dir {R}/calib_ncm_m"
]
for c in CMDS:
    c = c.format(R=R, D=D) + ' --data_dir /content/ts'
    print('>>>', c, flush=True)
    t = time.time()
    p = subprocess.run(c, shell=True)
    print(f'<<< exit {p.returncode} after {(time.time()-t)/60:.1f} min', flush=True)
    if p.returncode != 0:
        raise SystemExit('command failed; fix and re-run this cell (it resumes)')

In [ ]:
out = f'{DRIVE}/results/calib_t4'; os.makedirs(out, exist_ok=True)
for c in CMDS:
    rd = c.split('--run_dir ')[1].split()[0].format(R=R)
    name = os.path.basename(rd)
    for f in ['result.json', 'metrics.jsonl', 'config.yaml']:
        if os.path.exists(f'{rd}/{f}'):
            shutil.copy(f'{rd}/{f}', f'{out}/{name}__{f}')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
json.dump({'run': 'calib_t4', 'gpu': gpu, 'finished': time.ctime()}, open(f'{out}/_meta.json', 'w'))
print('Results copied to', out, '-> please drop this folder into the repo under results/colab/')